# Multithreading & Multiprocessing

Deep dive into concurrent and parallel execution in Python.


## When to Use Threads vs Processes

- **Threads**: I/O-bound tasks (network, disk, database)
- **Processes**: CPU-bound tasks (computation, number crunching)
- Python threads share memory but are limited by the GIL
- Processes have separate memory but bypass the GIL


In [ ]:
import threading
import multiprocessing
import time
import os



## Creating and Starting Threads


In [ ]:
def worker(name):
    print(f"  Thread {name} starting (PID: {os.getpid()})")
    time.sleep(1)
    print(f"  Thread {name} finished")


threads = []
for i in range(3):
    t = threading.Thread(target=worker, args=(i,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()
print("All threads done")


## Thread Synchronization with Lock

Prevents race conditions when multiple threads access shared data.


In [ ]:
counter = 0
lock = threading.Lock()

def increment():
    global counter
    for _ in range(100000):
        with lock:
            counter += 1

t1 = threading.Thread(target=increment)
t2 = threading.Thread(target=increment)
t1.start(); t2.start()
t1.join(); t2.join()
print(f"Counter with lock: {counter}")


## Race Condition Example (Without Lock)


In [ ]:
counter2 = 0

def increment_unsafe():
    global counter2
    for _ in range(100000):
        counter2 += 1  # Not atomic!

t1 = threading.Thread(target=increment_unsafe)
t2 = threading.Thread(target=increment_unsafe)
t1.start(); t2.start()
t1.join(); t2.join()
print(f"Counter without lock (unpredictable): {counter2}")


## Daemon Threads

Daemon threads run in the background and are killed automatically when the main thread exits.


In [ ]:
def background_monitor():
    """Daemon thread runs in background."""
    while True:
        print("  [Monitor] Checking system...")
        time.sleep(1)


daemon = threading.Thread(target=background_monitor, daemon=True)
daemon.start()

time.sleep(2.5)
print("Main thread exiting - daemon will be killed automatically")


## ThreadPoolExecutor (concurrent.futures)

Higher-level interface for managing threads - easier than raw `threading`.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch_url(url):
    time.sleep(0.5)  # Simulate I/O
    return f"Fetched {url}"

urls = [f"http://example.com/page{i}" for i in range(5)]

with ThreadPoolExecutor(max_workers=3) as executor:
    future_to_url = {executor.submit(fetch_url, url): url for url in urls}
    for future in as_completed(future_to_url):
        result = future.result()
        print(result)


## Multiprocessing Basics

Processes run in separate memory spaces and bypass the GIL.


In [ ]:
def cpu_heavy(n):
    """Simulate CPU-intensive work."""
    total = 0
    for i in range(n):
        total += i ** 2
    return f"Process {os.getpid()} computed: {total}"


if __name__ == "__main__":
    with multiprocessing.Pool(processes=4) as pool:
        results = pool.map(cpu_heavy, [10_000_000, 10_000_000, 10_000_000])
        for r in results:
            print(r)


## Synchronizing Processes with Queue


In [ ]:
def producer(q):
    for i in range(5):
        q.put({'item': i, 'value': i ** 2})
        time.sleep(0.2)
    q.put(None)  # Sentinel


def consumer(q):
    while True:
        data = q.get()
        if data is None:
            break
        print(f"  Consumed: {data}")


if __name__ == "__main__":
    q = multiprocessing.Queue()
    p1 = multiprocessing.Process(target=producer, args=(q,))
    p2 = multiprocessing.Process(target=consumer, args=(q,))
    p1.start(); p2.start()
    p1.join(); p2.join()


## Comparing Thread vs Process Performance

Thread vs process for CPU-bound and I/O-bound tasks.


In [ ]:
import timeit

def cpu_task(n=5_000_000):
    return sum(i * i for i in range(n))

def io_task():
    time.sleep(0.1)
    return 42


# CPU-bound test
if __name__ == "__main__":
    import threading
    import multiprocessing
    from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

    print("CPU-bound task (sum of squares)")
    threads = ThreadPoolExecutor(max_workers=4)
    processes = ProcessPoolExecutor(max_workers=4)

    start = time.time()
    list(threads.map(cpu_task, [5_000_000]*4))
    print(f"  Threads: {time.time() - start:.2f}s")

    start = time.time()
    list(processes.map(cpu_task, [5_000_000]*4))
    print(f"  Processes: {time.time() - start:.2f}s")


## Interview Problem 1: Thread-Safe Singleton


**Problem:** Implement a thread-safe singleton pattern using a lock.


In [ ]:
import threading

class ThreadSafeSingleton:
    """Thread-safe singleton using double-checked locking."""
    _instance = None
    _lock = threading.Lock()

    def __new__(cls):
        if cls._instance is None:
            with cls._lock:
                if cls._instance is None:
                    cls._instance = super().__new__(cls)
        return cls._instance


# Test
def create_singleton():
    s = ThreadSafeSingleton()
    print(f"  {id(s)}")

threads = [threading.Thread(target=create_singleton) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()


## Interview Problem 2: Producer-Consumer (Bounded Buffer)

**Problem:** Implement a producer-consumer pattern with a bounded buffer using threading primitives.


In [ ]:
import threading
import time
import random

class BoundedBuffer:
    def __init__(self, capacity):
        self.buffer = []
        self.capacity = capacity
        self.lock = threading.Lock()
        self.not_full = threading.Condition(self.lock)
        self.not_empty = threading.Condition(self.lock)

    def put(self, item):
        with self.not_full:
            while len(self.buffer) >= self.capacity:
                self.not_full.wait()
            self.buffer.append(item)
            print(f"  Put: {item} (size={len(self.buffer)})")
            self.not_empty.notify()

    def get(self):
        with self.not_empty:
            while len(self.buffer) == 0:
                self.not_empty.wait()
            item = self.buffer.pop(0)
            print(f"  Got: {item} (size={len(self.buffer)})")
            self.not_full.notify()
            return item


def producer(buf, n):
    for i in range(n):
        buf.put(i)
        time.sleep(random.uniform(0.05, 0.15))

def consumer(buf, n):
    for _ in range(n):
        buf.get()
        time.sleep(random.uniform(0.08, 0.12))


buf = BoundedBuffer(3)
prod = threading.Thread(target=producer, args=(buf, 10))
cons = threading.Thread(target=consumer, args=(buf, 10))
prod.start(); cons.start()
prod.join(); cons.join()


## Interview Problem 3: Parallel Web Scraper (Simulated)

**Problem:** Simulate scraping multiple URLs concurrently. Compare sequential vs parallel performance.


In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def simulate_fetch(url):
    """Simulate HTTP request with delay."""
    delay = hash(url) % 3 + 1  # 1-3 second delay
    time.sleep(delay)
    return f"{url} -> {delay}s"


urls = [f"https://site{i}.com/data" for i in range(8)]

# Sequential
start = time.time()
results_seq = [simulate_fetch(url) for url in urls]
seq_time = time.time() - start

# Parallel
start = time.time()
with ThreadPoolExecutor(max_workers=4) as executor:
    results_par = list(executor.map(simulate_fetch, urls))
par_time = time.time() - start

print(f"Sequential: {seq_time:.2f}s")
print(f"Parallel:   {par_time:.2f}s")
print(f"Speedup:    {seq_time / par_time:.1f}x")


## Interview Problem 4: Gather Partial Results (Reduce Pattern)

**Problem:** Split a large computation across threads, then combine partial results.


In [ ]:
import threading
import random

partial_sums = []
lock = threading.Lock()

def partial_sum(data_chunk):
    total = sum(data_chunk)
    with lock:
        partial_sums.append(total)

data = [random.randint(1, 100) for _ in range(1000)]
n_threads = 4
chunk_size = len(data) // n_threads
chunks = [data[i:i+chunk_size] for i in range(0, len(data), chunk_size)]

threads = [threading.Thread(target=partial_sum, args=(chunk,)) for chunk in chunks]
for t in threads: t.start()
for t in threads: t.join()

total = sum(partial_sums)
print(f"Total from threads: {total}")
print(f"Verification:       {sum(data)}")


## Key Takeaways

- Use `threading` for I/O-bound tasks, `multiprocessing` for CPU-bound
- Synchronize shared state with `Lock`, `RLock`, `Semaphore`
- `ThreadPoolExecutor`/`ProcessPoolExecutor` are simpler than raw thread/process management
- Daemon threads auto-terminate when the main thread exits
- The GIL prevents true parallelism in threads for CPU-bound work
- Processes have independent memory - use `Queue` or `Pipe` for IPC
